# Dataset exploration

Load the SemEval-2014 Task 4 laptop and restaurant XML files from `data/raw/` (see `data/README.md`) and inspect them.

Status: only the official **trial files** (100 sentences per domain) are reachable from this environment; the full train/test files require a Metashare account (see `data/README.md`). Every number below comes from running the code in this notebook against the trial files, not from assumption or from the task papers. Whoever adds the full files should rerun this notebook and update the numbers.


In [ ]:
from pathlib import Path

from absa.data import normalize_sentences, parse_laptop_xml, parse_restaurant_xml

laptop_sentences = parse_laptop_xml(Path("../data/raw/laptops-trial.xml"))
restaurant_sentences, restaurant_categories = parse_restaurant_xml(
    Path("../data/raw/restaurants-trial.xml")
)

laptop_records, laptop_no_aspect = normalize_sentences(laptop_sentences, domain="laptop")
restaurant_records, restaurant_no_aspect = normalize_sentences(
    restaurant_sentences, domain="restaurant"
)
all_records = laptop_records + restaurant_records

print("laptop sentences:", len(laptop_sentences), "| aspect records:", len(laptop_records))
print(
    "restaurant sentences:",
    len(restaurant_sentences),
    "| aspect records:",
    len(restaurant_records),
)

## Sentiment label distribution

Measured result (trial files): laptop `{'positive': 29, 'negative': 16, 'neutral': 4}`, restaurant `{'positive': 68, 'negative': 18, 'neutral': 10}`. No aspect term in either trial file is labelled `conflict`; `docs/dataset_annotation.md` has a real example of `conflict` at the aspect-category level instead. Both domains are positive-skewed, restaurant more so.


In [ ]:
from absa.data import class_balance

print("laptop:", class_balance(laptop_records))
print("restaurant:", class_balance(restaurant_records))

## Aspect frequency

Measured result (trial files, both domains combined), most frequent first: `food` (16), `price` (7), `service` (4), `prices` (4), `battery life` (3), `Service` (3), `screen` (2), `performance` (2), `features` (2), `speed` (2).

Data-quality observation: `service` and `Service` are counted as different aspects because aspect terms are not case-normalised, and `price`/`prices` similarly split a singular/plural form of the same aspect. This is expected given issue #17's scope (it normalises label spelling, not aspect-term spelling), and is worth keeping in mind for aspect extraction (#30-#32), where near-duplicate surface forms may need grouping.


In [ ]:
from absa.data import aspect_frequency

aspect_frequency(all_records, top_n=10)

## Review length and sentences without aspects

Measured result (trial files, both domains combined): review character length ranges 15-193, mean ~76.1. 60/100 laptop sentences and 46/100 restaurant sentences carry no aspect-term annotation at all (see `docs/dataset_annotation.md`); `normalize_sentences` keeps these in a separate `..._no_aspect.jsonl` file rather than dropping them.


In [ ]:
from absa.data import review_lengths

lengths = review_lengths(all_records)
print("min/max/mean review length:", min(lengths), max(lengths), sum(lengths) / len(lengths))
print("laptop sentences with no aspect term:", len(laptop_no_aspect), "/", len(laptop_sentences))
print(
    "restaurant sentences with no aspect term:",
    len(restaurant_no_aspect),
    "/",
    len(restaurant_sentences),
)

## Train/validation split sanity check

`split_train_val` groups by the normalised review text (so different sentence ids that carry identical text stay together), then `assert_no_leakage` checks it. On the laptop trial records with `val_fraction=0.2, seed=42`: 41 train records, 8 validation records, zero overlap.

This is a sanity check on trial-scale data, not the final split: the real validation split (#19's actual deliverable for training) must be carved from the full official training file once it is available, with the same seeded, grouped method.


In [ ]:
from absa.data import split_train_val
from absa.data.leakage import assert_no_leakage

train, val = split_train_val(laptop_records, val_fraction=0.2, seed=42)
assert_no_leakage(train, val)
print("train:", len(train), "val:", len(val))

## Summary

- Both domains are positive-skewed; `conflict` exists in the schema and in restaurant categories but was not observed on any laptop or restaurant aspect **term** in the trial sample.
- Aspect terms are highly varied and not case- or plural-normalised; a handful of aspects (`food`, `price`/`prices`, `service`/`Service`) dominate the restaurant counts.
- Close to half of all sentences carry no aspect annotation; these are kept, not dropped, in the no-aspect files.
- The grouped, seeded split keeps every sentence's records together and is leakage-free on this sample; the same method should be applied to the full training file.
- All numbers above come from the 100-sentence trial files, not the full SemEval-2014 benchmark (~3000 sentences per domain); rerun this notebook once the full files are available (see `data/README.md`) and update this summary.
